In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from nodepy import rk, ivp
from scipy.optimize import fsolve
from ipywidgets import interact, Dropdown, FloatText

In [ ]:
def linear_oscillator(t,u):
    du0 = u[1]
    du1 = -u[0]
    return np.array((du0, du1))

u0 = np.array([1.,0.])
t0 = 0.; T = 100.
lin_osc = ivp.IVP(linear_oscillator,u0,t0,T)

def nonlinear_oscillator(t,u):
    usq = u[0]**2 + u[1]**2
    du0 = u[1]/usq
    du1 = -u[0]/usq
    return np.array((du0, du1))

u0 = np.array([1.,0.])
t0 = 0.; T = 100.
nl_osc = ivp.IVP(nonlinear_oscillator,u0,t0,T)

def pendulum(t,u):
    du0 = u[1]
    du1 = -np.sin(u[0])
    return np.array((du0, du1))

u0 = np.array([1.,0.])
t0 = 0.; T = 100.
pend = ivp.IVP(pendulum,u0,t0,T)

In [ ]:
def plot_solution(tt, uu, title=None):
    uu = np.array(uu)
    fig, ax = plt.subplots(1, 3, figsize=(12,4))

    ax[0].plot(tt,uu)
    ax[1].plot(uu[:,0],uu[:,1])
    ax[1].set_xlim(-2,2); ax[1].set_ylim(-2,2)

    energy = uu[:,0]**2 + uu[:,1]**2
    ax[2].plot(tt,energy)
    if title:
        fig.suptitle(title)

In [ ]:
def be_solve(ivp, dt=0.1):
    f = ivp.rhs
    nsteps = int(np.ceil((ivp.T-ivp.t0)/dt))
    yvals = [ivp.u0]
    tvals = np.linspace(t0,ivp.T,nsteps+1)
    for i in range(nsteps):
        func = lambda u: u - dt*f(tvals[i],u) - yvals[-1]
        root, info, status, msg = fsolve(func, yvals[-1], full_output=True)
        yvals.append(root)
    return tvals, yvals

def fe_solve(ivp, dt=0.1):
    f = ivp.rhs
    nsteps = int(np.ceil((ivp.T-ivp.t0)/dt))
    yvals = [ivp.u0]
    tvals = np.linspace(t0,ivp.T,nsteps+1)
    for i in range(nsteps):
        ynew = yvals[-1] + dt*f(tvals[-1],yvals[-1])
        yvals.append(ynew)
    return tvals, yvals

def imp_mid_solve(ivp, dt=0.1):
    f = ivp.rhs
    nsteps = int(np.ceil((ivp.T-ivp.t0)/dt))
    yvals = [ivp.u0]
    tvals = np.linspace(t0,ivp.T,nsteps+1)
    for i in range(nsteps):
        tmid = 0.5*(tvals[i]+tvals[i-1])
        func = lambda u: u - yvals[-1] - dt*f(tvals[i],0.5*(u+yvals[-1])) 
        root, info, status, msg = fsolve(func, yvals[-1], full_output=True)
        yvals.append(root)
    return tvals, yvals

rk4 = rk.loadRKM("RK44")

In [ ]:
problems = { "linear oscillator": lin_osc,
            "nonlinear oscillator": nl_osc,
            "pendulum": pend}

methods = { "backward Euler": be_solve,
            "forward Euler": fe_solve,
            "implicit midpoint": imp_mid_solve,
            "RK4": rk4.__call__}

def test_method(prob,meth,dt=0.1,T=10.):
    prob.T = T
    tt, uu = meth(prob,dt=dt)
    plot_solution(tt, uu)

In [ ]:
interact(test_method, prob=problems, meth=methods, dt=(0.1,4,0.1), T=[1,10,100,1000]);